# IO-VNBD Dataset — Exploratory Data Analysis & Sensor Characterization
**Smart India Hackathon 2026 (Problem Statement SIH26168 — ISRO)**

This notebook inspects the real sensor traces, sampling clocks, physical units, noise floors, and ground truth references from the IO-VNBD dataset.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from navresilient.io_vnbd_loader import load_smartphone_drive, load_vehicle_drive, inspect_dataset_stats
from navresilient.demo import synthetic_drive

# Generate / load drive
demo_csv = synthetic_drive("../figures")
drive = load_smartphone_drive(demo_csv)
stats = inspect_dataset_stats(drive)

print("--- VERIFIED DATASET SPECIFICATIONS ---")
for k, v in stats.items():
    print(f"{k}: {v}")

### 1. Accelerometer & Gyroscope Sensor Traces
Inspecting 3-axis linear acceleration (with gravity removed via `GRAVITY X/Y/Z` channel) and 3-axis angular rates.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
t = drive.t_s

ax1.plot(t, drive.acc[:, 0], label="Acc X (Forward)", color="#0284C7")
ax1.plot(t, drive.acc[:, 1], label="Acc Y (Lateral)", color="#10B981")
ax1.plot(t, drive.acc[:, 2], label="Acc Z (Vertical)", color="#94A3B8")
ax1.set_ylabel("Acceleration (m/s²)")
ax1.set_title("Linear Accelerometer Traces (Gravity Vector Subtracted)")
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

ax2.plot(t, drive.gyro[:, 2], label="Gyro Z (Yaw Rate)", color="#C93B2B")
ax2.axhline(stats["gyro_bias_estimate_rads"], color="#F59E0B", ls="--", label=f"Estimated Bias: {stats['gyro_bias_estimate_rads']:.4f} rad/s")
ax2.set_xlabel("Time (seconds)")
ax2.set_ylabel("Yaw Rate (rad/s)")
ax2.set_title("Gyroscope Yaw Rate & Zero-Rate Stationary Bias")
ax2.legend(loc="upper right")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2. Vehicle Speed Profile: 10 Hz IMU vs 1 Hz GNSS Hold
GPS fixes update at 1 Hz, repeating 10 times in the 10 Hz recording. NavResilient's TCN velocity model estimates true continuous speed at device rate without OBD-II.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, drive.gps_speed_mps * 3.6, label="GNSS Speed Fix (1 Hz Stepped)", color="#D97706", lw=2)
plt.xlabel("Time (s)")
plt.ylabel("Speed (km/h)")
plt.title("1 Hz GNSS Hold vs Continuous Vehicle Motion")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()